In [0]:
%python
from datetime import datetime

# ==========================================
# ARCHIVE FUNCTION
# ==========================================

def archive_old_files(entity_folder, archive_path):

    files = dbutils.fs.ls(entity_folder)

    csv_files = [
        f for f in files
        if not f.isDir()
    ]

    if len(csv_files) <= 1:
        print(f"ℹ️ No old files to archive in {entity_folder}")
        return

    # ==========================================
    # SORT FILES BY MODIFICATION TIME
    # ==========================================

    sorted_files = sorted(
        csv_files,
        key=lambda x: x.modificationTime,
        reverse=True
    )

    latest_file = sorted_files[0]

    print(f"\n✅ Keeping latest file:")
    print(f"   {latest_file.name}")

    old_files = sorted_files[1:]

    archive_folder = datetime.now().strftime('%Y-%m-%d')

    target_archive_dir = (
        f"{archive_path}{archive_folder}/"
    )

    # ==========================================
    # MOVE OLD FILES
    # ==========================================

    for file in old_files:

        destination = (
            f"{target_archive_dir}{file.name}"
        )

        dbutils.fs.mv(
            file.path,
            destination
        )

        print(f"📦 Archived: {file.name}")

# ==========================================
# S3 PATHS
# ==========================================

S3_BUCKET = "s3://retail-sales-datawarehouse-shivanand-409953608511-eu-north-1-an"

archive_raw = f"{S3_BUCKET}/archive/raw/"

# ==========================================
# EXECUTE FOR EACH ENTITY
# ==========================================

archive_old_files(
    f"{S3_BUCKET}/raw/customers/",
    archive_raw
)

archive_old_files(
    f"{S3_BUCKET}/raw/product/",
    archive_raw
)

archive_old_files(
    f"{S3_BUCKET}/raw/store/",
    archive_raw
)

archive_old_files(
    f"{S3_BUCKET}/raw/sales/",
    archive_raw
)

print("\n✅ Archive Process Completed Successfully!")